# GPU Network Agent — Fine-Tuning Test

This notebook runs the GPU agent on Colab's T4 GPU, connecting to your production server at `34.18.164.66:8000`.

**Setup:** Runtime → Change runtime type → **T4 GPU**

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install -q websockets httpx pynvml psutil huggingface-hub \
    torch transformers peft bitsandbytes trl datasets accelerate

## 3. Clone the Repo

In [ ]:
!git clone https://github.com/kmalarifi97/training-network.git /content/gpunetwork
!cd /content/gpunetwork && git checkout dev

## 4. Register a Test Cafe & Get API Key

This creates a new cafe for the Colab agent. **Save the API key from the output.**

In [ ]:
import requests, json

SERVER = "http://34.18.164.66:8000"

resp = requests.post(f"{SERVER}/cafes/register", json={
    "name": "Colab Test GPU",
    "owner_name": "Khalid",
    "location": "Google Colab"
})
creds = resp.json()
print(json.dumps(creds, indent=2))

CAFE_ID = creds["cafe_id"]
API_KEY = creds["api_key"]
print(f"\nCAFE_ID: {CAFE_ID}")
print(f"API_KEY: {API_KEY}")

## 5. Configure the Agent

Write a config that points to your production server and skips idle detection (no mouse/keyboard on Colab).

In [ ]:
import json, os

config = {
    "server_url": "ws://34.18.164.66:8000/agents/connect",
    "cafe_id": CAFE_ID,
    "api_key": API_KEY,
    "idle_threshold_minutes": 0,
    "heartbeat_interval_seconds": 10,
    "model_cache_dir": "/content/models",
    "log_dir": "/content/logs",
    "max_gpu_usage_percent": 95,
    "max_ram_usage_percent": 90,
    "max_disk_usage_gb": 100
}

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/logs", exist_ok=True)

with open("/content/gpunetwork/agent/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config written:")
print(json.dumps(config, indent=2))

## 6. Patch Idle Detector for Linux

The idle detector uses Win32 APIs. On Colab (Linux), we patch it to always report idle (= always available for jobs).

In [ ]:
patch = '''"""Idle detector — Colab/Linux stub (always idle)."""

class IdleDetector:
    def __init__(self, idle_threshold_minutes=0):
        self.idle_threshold_minutes = idle_threshold_minutes

    def is_user_active(self) -> bool:
        return False  # Always idle — ready for jobs

    def get_status(self) -> dict:
        return {
            "idle_seconds": 9999,
            "user_active": False,
            "threshold_minutes": self.idle_threshold_minutes,
        }
'''

with open("/content/gpunetwork/agent/idle_detector.py", "w") as f:
    f.write(patch)

print("Idle detector patched for Linux (always idle)")

## 7. Start the Agent

The agent will:
1. Connect to your server via WebSocket
2. Report AVAILABLE
3. Wait for jobs

**Keep this cell running.** Then go to http://34.18.164.66:8000/ and upload your JSONL in the Fine-Tuning tab.

The agent will pick up the job automatically.

In [ ]:
import subprocess, sys, os

os.chdir("/content/gpunetwork/agent")

# Run the agent — output streams live
process = subprocess.Popen(
    [sys.executable, "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    process.terminate()
    print("\n--- Agent stopped ---")

## 8. (Optional) Upload Training Data via API

If you prefer to start the fine-tune from this notebook instead of the web UI:

In [ ]:
# Upload your JSONL file to the server and start fine-tuning
# First, upload the file to Colab (use the file browser on the left),
# or use this cell to upload from your machine:

from google.colab import files
uploaded = files.upload()  # This will prompt you to pick a file

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

In [ ]:
# Submit the fine-tune job
import requests

SERVER = "http://34.18.164.66:8000"

with open(filename, "rb") as f:
    resp = requests.post(
        f"{SERVER}/finetune/upload",
        files={"file": (filename, f, "application/jsonl")},
        data={
            "base_model": "tinyllama-1.1b",
            "epochs": 3,
            "batch_size": 4,
            "learning_rate": 0.0002,
            "lora_r": 16,
            "lora_alpha": 32,
        }
    )

result = resp.json()
print(json.dumps(result, indent=2))
print(f"\nTrack at: {SERVER}/finetune/{result.get('run_id', '')}")